# 03 — Benchmarks

Rigorous timing with `BenchmarkTools.jl`.

**Topics covered**
- `@btime` / `@benchmark` for single-run and Monte Carlo timings
- Memory allocation comparison
- Scaling experiment: runtime vs. number of trials
- Speedup table

In [10]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()

include("../src/types.jl")
include("../src/generate_data.jl")
include("../src/serial_sim.jl")
include("../src/parallel_sim.jl")
include("../src/monte_carlo.jl")
include("../src/benchmarks.jl")

using BenchmarkTools, Printf
println("Julia threads: ", Threads.nthreads())

  Activating project at `~/Documents/MIT/18.337/parallel-school-assignment-sim`


Julia threads: 8


## 1. Generate Scenario

In [11]:
students, schools = generate_scenario(; seed = 42)

(Student[Student(1, [17, 2, 20, 13, 15], 0.5776180399036042), Student(2, [20, 15, 2, 18, 13], 0.42108744744559745), Student(3, [7, 20, 6, 13, 1], 0.1189646291016957), Student(4, [11, 16, 18, 10, 4], 0.5084067760768716), Student(5, [1, 11, 14, 20, 5], 0.06789763512912672), Student(6, [1, 13, 10, 5, 16], 0.4272368880629398), Student(7, [13, 2, 10, 18, 14], 0.25466424071119564), Student(8, [13, 2, 17, 3, 12], 0.5569127441511577), Student(9, [18, 19, 20, 9, 17], 0.3414258240267456), Student(10, [13, 5, 2, 4, 7], 0.7067411881639054)  …  Student(9991, [3, 18, 7, 11, 20], 0.9558741362169305), Student(9992, [20, 7, 2, 4, 13], 0.5953170920809927), Student(9993, [19, 6, 11, 15, 20], 0.5921289844392843), Student(9994, [10, 20, 7, 16, 2], 0.71127422924206), Student(9995, [18, 7, 16, 6, 2], 0.7493663962591), Student(9996, [16, 6, 13, 15, 2], 0.6234351111983603), Student(9997, [11, 13, 17, 4, 20], 0.6666049133687935), Student(9998, [10, 14, 17, 13, 11], 0.3971050817557533), Student(9999, [19, 14, 5,

## 2. Quick Timing with @btime

In [12]:
# Single serial assignment
@btime run_serial($students, $schools);

  642.667 μs (15 allocations: 560.98 KiB)


In [13]:
# 50-trial serial MC
@btime run_serial_mc($students, $schools, 50);

  38.819 ms (1024 allocations: 39.14 MiB)


In [14]:
# 50-trial parallel MC
@btime run_parallel_mc($students, $schools, 50);

  6.873 ms (2146 allocations: 40.11 MiB)


## 3. Full Benchmark Suite

In [15]:
bench = run_all_benchmarks(students, schools; n_trials = 50, n_samples = 5)

Benchmarking single serial assignment …
Benchmarking serial Monte Carlo (50 trials) …
Benchmarking parallel Monte Carlo (50 trials) …

Benchmark Results
  Julia threads available: 8
  Monte Carlo trials     : 50
  Serial (single run)                 0.78 ms     0.57 MB
  Serial MC                          42.53 ms    41.04 MB
  Parallel MC                         9.33 ms    42.06 MB
  Parallel speedup                    4.56x


(serial_single = Trial(658.916 μs), serial_mc = Trial(38.834 ms), parallel_mc = Trial(8.810 ms), speedup = 4.558248088468238)

## 4. Scaling Experiment

In [16]:
trial_counts, serial_times, parallel_times = scaling_experiment(
    students, schools;
    trial_counts = [10, 25, 50, 100, 200],
    n_samples    = 3,
)

  n_trials = 10
  n_trials = 25
  n_trials = 50
  n_trials = 100
  n_trials = 200


([10, 25, 50, 100, 200], [8.097042, 19.012, 41.371792, 86.450125, 194.487292], [1.686792, 4.545292, 10.573833, 18.585625, 38.725791])

In [17]:
println("n_trials | Serial (ms) | Parallel (ms) | Speedup")
println("-" ^ 48)
for i in eachindex(trial_counts)
    sp = serial_times[i] / parallel_times[i]
    @printf("%8d | %11.2f | %13.2f | %.2fx\n", trial_counts[i], serial_times[i], parallel_times[i], sp)
end

n_trials | Serial (ms) | Parallel (ms) | Speedup
------------------------------------------------
      10 |        8.10 |          1.69 | 4.80x
      25 |       19.01 |          4.55 | 4.18x
      50 |       41.37 |         10.57 | 3.91x
     100 |       86.45 |         18.59 | 4.65x
     200 |      194.49 |         38.73 | 5.02x
